In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

#### Pivoting: adding missing days, fixing NaNs with zeros and sums

In [2]:
def pivot_df(df_imputed, imputed_vars, name='median'):

    # ids and dates
    ids = df_imputed['id'].unique()
    df_imputed['date'] = pd.to_datetime(df_imputed['date']) 

    # summing
    sum_vars = [v for v in df_imputed['variable'].unique() if v not in imputed_vars]  
    median_group = df_imputed[df_imputed['variable'].isin(imputed_vars)].copy()
    median_agg = (median_group.groupby(['id', 'date', 'variable'])['value'].median().reset_index())

    sum_group = df_imputed[df_imputed['variable'].isin(sum_vars)].copy()
    sum_agg = (sum_group.groupby(['id', 'date', 'variable'])['value'].sum().reset_index())

    # pivot and combine
    combined = pd.concat([median_agg, sum_agg], ignore_index=True)
    daily_df = combined.pivot(index=['id', 'date'], columns='variable', values='value').reset_index()
    daily_df.columns.name = None

    # fill in 0 for sumtype variables
    all_columns = daily_df.columns.tolist()
    cols_to_fill = [col for col in all_columns if col not in imputed_vars and col not in ['id', 'date']]
    daily_df[cols_to_fill] = daily_df[cols_to_fill].fillna(0)

    # missing dates per used added
    all_dates = pd.date_range(daily_df['date'].min(), daily_df['date'].max())
    full_index = pd.MultiIndex.from_product([ids, all_dates], names=['id', 'date'])
    daily_df_full = daily_df.set_index(['id', 'date']).reindex(full_index).reset_index()

    print(f"{name} dataset: shape after pivoting = {daily_df_full.shape}")
    display(daily_df_full.head())
    return daily_df_full


In [3]:
# datatsets from cleaning
df_median_imputed = pd.read_csv('df_median.csv')
df_kalman_imputed = pd.read_csv('df_kalman.csv')

# imputed variables
imputed_vars = ['mood', 'circumplex.arousal', 'circumplex.valence', 'activity']

# use function made previously
daily_df_median_full = pivot_df(df_median_imputed, imputed_vars, name='median')
daily_df_kalman_full = pivot_df(df_kalman_imputed, imputed_vars, name='kalman')

# to compare later shapes
print("Before trimming:")
print("median shape:", daily_df_median_full.shape)
print("kalman shape:", daily_df_kalman_full.shape)

median dataset: shape after pivoting = (3051, 21)


,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-17,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,NaN,NaN,NaN,0.0,0.0
1,AS14.01,2014-02-18,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,0.0,0.0
2,AS14.01,2014-02-19,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,7.0,NaN,NaN,NaN,0.0,2.0
3,AS14.01,2014-02-20,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,NaN,NaN,NaN,0.0,3.0
4,AS14.01,2014-02-21,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,1.0


kalman dataset: shape after pivoting = (3051, 21)


,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-17,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,NaN,NaN,NaN,0.0,0.0
1,AS14.01,2014-02-18,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,0.0,0.0
2,AS14.01,2014-02-19,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,7.0,NaN,NaN,NaN,0.0,2.0
3,AS14.01,2014-02-20,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,NaN,NaN,NaN,0.0,3.0
4,AS14.01,2014-02-21,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,1.0


Before trimming:
median shape: (3051, 21)
kalman shape: (3051, 21)


#### Trimming the dataset till last known mood entry and trimming the beginning 5 days for prediction dataset

In [4]:
# trimming till last known mood for median method
if 'mood' in daily_df_median_full.columns:
    last_mood_dates = (
        daily_df_median_full[~daily_df_median_full['mood'].isna()]
        .groupby('id')['date'].max()
    )

    daily_df_median_full = daily_df_median_full.merge(
        last_mood_dates.rename('last_mood_date'), on='id', how='left'
    )
    daily_df_median_full = daily_df_median_full[daily_df_median_full['date'] <= daily_df_median_full['last_mood_date']]
    daily_df_median_full.drop(columns='last_mood_date', inplace=True)


# trimming till last known mood for kalman method
if 'mood' in daily_df_kalman_full.columns:
    last_mood_dates = (
        daily_df_kalman_full[~daily_df_kalman_full['mood'].isna()]
        .groupby('id')['date'].max()
    )

    daily_df_kalman_full = daily_df_kalman_full.merge(
        last_mood_dates.rename('last_mood_date'), on='id', how='left'
    )
    daily_df_kalman_full = daily_df_kalman_full[daily_df_kalman_full['date'] <= daily_df_kalman_full['last_mood_date']]
    daily_df_kalman_full.drop(columns='last_mood_date', inplace=True)

print("\nAfter trimming:")
print("median shape:", daily_df_median_full.shape)
print("kalman shape:", daily_df_kalman_full.shape)



After trimming:
median shape: (2225, 21)
kalman shape: (2225, 21)


In [5]:
# trimming first 5 days, for prediction dataset
# median method set
daily_df_median_full = daily_df_median_full.sort_values(['id', 'date']).copy()
daily_df_median_full['day_index'] = daily_df_median_full.groupby('id').cumcount()
daily_df_median_full = daily_df_median_full[daily_df_median_full['day_index'] >= 5]
daily_df_median_full.drop(columns='day_index', inplace=True)

# kalman method set
daily_df_kalman_full = daily_df_kalman_full.sort_values(['id', 'date']).copy()
daily_df_kalman_full['day_index'] = daily_df_kalman_full.groupby('id').cumcount()
daily_df_kalman_full = daily_df_kalman_full[daily_df_kalman_full['day_index'] >= 5]
daily_df_kalman_full.drop(columns='day_index', inplace=True)

In [ ]:
print("After trimming first 5 days:")
print("median shape:", daily_df_median_full.shape)
print("kalman shape:", daily_df_kalman_full.shape)

user_counts = daily_df_median_full.groupby('id').size()
print("Users with <6 days:", (user_counts < 6).sum())

After trimming first 5 days:
median shape: (2090, 21)
kalman shape: (2090, 21)
Users with <6 days: 0


27 users × 5 days = 135 rows dropped